In [ ]:
# !pip install fast-plate-ocr[train,onnx]

In [11]:
!tree dataset/lp_crops_balanced/ --filelimit 5

dataset/lp_crops_balanced/
├── train  [21980 entries exceeds filelimit, not opening dir]
└── val  [5386 entries exceeds filelimit, not opening dir]

2 directories, 0 files


In [24]:
from pathlib import Path

def generate_annotations_csv(
    images_dir: str,
    output_csv: str,
    image_ext: str = "*.jpg"
):
    images_dir = Path(images_dir)
    output_csv = Path(output_csv)

    lines = ["image_path,plate_text"]

    for image_path in images_dir.glob(image_ext):
        label_path = image_path.with_suffix(".txt")

        if not image_path.exists() or not label_path.exists():
            continue

        # keep only: train/xxx.jpg
        image_path_simplified = image_path.parent.name + "/" + image_path.name

        with label_path.open() as f:
            label = f.readline().strip().upper()
            if ("\"" in label) or (" " in label):
                continue

        lines.append(f"{image_path_simplified},{label}")

    output_csv.write_text("\n".join(lines))

generate_annotations_csv(
    images_dir="./dataset/lp_crops_balanced/train",
    output_csv="./dataset/lp_crops_balanced/train_annotations.csv"
)

generate_annotations_csv(
    images_dir="./dataset/lp_crops_balanced/val",
    output_csv="./dataset/lp_crops_balanced/val_annotations.csv"
)

In [50]:
# Check the train annotations
!fast-plate-ocr validate-dataset \
  --annotations-file ./dataset/lp_crops_balanced/train_annotations.csv \
  --plate-config-file ./config/cct_s_v2_global_plate_config.yaml

I0000 00:00:1777390333.370162 3163977 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777390333.399918 3163977 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777390334.302312 3163977 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10988/10988 Scanning images Scanning images

 Validation S

In [33]:
# Check the train annotations
!fast-plate-ocr validate-dataset \
  --annotations-file ./dataset/lp_crops_balanced/val_annotations.csv \
  --plate-config-file ./config/cct_s_v2_global_plate_config.yaml

I0000 00:00:1777386740.497520 3157403 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777386740.528191 3157403 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777386741.425627 3157403 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2693/2693 Scanning images Scanning images

 Validation S

In [42]:
!fast-plate-ocr dataset-stats \
  --annotations ./dataset/lp_crops_balanced/train_annotations.csv \
  --plate-config-file ./config/cct_s_v2_global_plate_config.yaml

I0000 00:00:1777387000.233670 3157633 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777387000.265075 3157633 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777387001.164344 3157633 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
┌────────────────────────── Dataset Statistics ───────────────────────────┐
│                         Plate Len

In [45]:
from pathlib import Path
import math
import random
from PIL import Image
import matplotlib.pyplot as plt


def plot_dataset(
    images_dir: str,
    n: int = 24,
    cols: int = 6,
    figsize=(16, 10),
    random_sample: bool = True,
    image_ext: str = "*.jpg"
):
    images_dir = Path(images_dir)
    image_paths = list(images_dir.glob(image_ext))

    if len(image_paths) == 0:
        print("No images found.")
        return

    if random_sample:
        image_paths = random.sample(image_paths, min(n, len(image_paths)))
    else:
        image_paths = image_paths[:n]

    rows = math.ceil(len(image_paths) / cols)

    plt.figure(figsize=figsize)

    for idx, image_path in enumerate(image_paths):
        label_path = image_path.with_suffix(".txt")

        label = ""
        if label_path.exists():
            with label_path.open() as f:
                label = f.readline().strip()

        image = Image.open(image_path)

        ax = plt.subplot(rows, cols, idx + 1)
        ax.imshow(image)
        ax.axis("off")

        ax.set_title(label, fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_dataset(
    images_dir="./dataset/lp_crops_balanced/train",
    n=60,
    cols=10,
    random_sample=True
)